# Lab 02: Certificate Lifecycle — Scan, Renew, Status, Inventory

Run the **complete** lifecycle end-to-end:
```
SCAN -> RENEW -> CHECK STATUS -> DOWNLOAD (simulated) -> INVENTORY
```
All in mock mode.

> **Estimated time:** 20 minutes

## Load config

In [ ]:
# Config is written by the SageMaker lifecycle script from SSM at space startup.
# If this fails, re-launch the JupyterLab space to trigger the lifecycle script.
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm  = boto3.client('lambda',        region_name=AWS_REGION)
ddb = boto3.resource('dynamodb',    region_name=AWS_REGION)
sm  = boto3.client('secretsmanager',region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '💀', 'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Table   : {CERT_TABLE_NAME}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('✅ Environment ready')

## Phase 1 — Scan

In [ ]:
from tabulate import tabulate
from datetime import datetime, timezone
import time

# Clear table for clean demo
table = ddb.Table(CERT_TABLE_NAME)
resp = table.scan(ProjectionExpression='#d, order_id', ExpressionAttributeNames={'#d': 'domain'})
with table.batch_writer() as b:
    for item in resp.get('Items', []):
        b.delete_item(Key={'domain': item['domain'], 'order_id': item['order_id']})
print('Table cleared')

# Scan
result = invoke(LAMBDA_SCAN, {'use_mock': True, 'threshold_days': 60})
certs = result['certificates']
print(f'\nScan found {len(certs)} certs:')
for c_ in certs:
    print(f"  {PRIORITY_EMOJI.get(c_['priority'])} {c_['common_name']} — {c_['days_remaining']}d")

critical_certs = [c_ for c_ in certs if c_['priority'] in ('EXPIRED','CRITICAL','HIGH')]

## Phase 2 — Renew critical certs

In [ ]:
renewal_results = []
for cert in critical_certs:
    r = invoke(LAMBDA_RENEW, {
        'order_id': cert['order_id'], 'common_name': cert['common_name'],
        'sans': cert.get('sans', [cert['common_name']]), 'use_mock': True
    })
    print(f"  ✅ {cert['common_name']} -> new order: {r.get('new_order_id')}")
    renewal_results.append({'domain': cert['common_name'],
        'old_order': cert['order_id'], 'new_order': r.get('new_order_id')})

print(f'\nRenewals submitted: {len(renewal_results)}')

## Phase 3 — Check status & simulate issuance

In [ ]:
# Simulate issuance (in production, DigiCert sets this)
for rn in renewal_results:
    table.update_item(
        Key={'domain': rn['domain'], 'order_id': rn['old_order']},
        UpdateExpression='SET renewal_status = :s, issued_at = :t',
        ExpressionAttributeValues={':s': 'issued',
            ':t': datetime.now(timezone.utc).isoformat()}
    )
    print(f"  {rn['domain']} -> issued")
print('\nAll renewed certs marked as issued')

## Phase 4 — Simulate download & mark complete

In [ ]:
for rn in renewal_results:
    table.update_item(
        Key={'domain': rn['domain'], 'order_id': rn['old_order']},
        UpdateExpression='SET renewal_status = :s, downloaded_at = :t',
        ExpressionAttributeValues={':s': 'completed',
            ':t': datetime.now(timezone.utc).isoformat()}
    )
print('All certs marked completed')

## Phase 5 — Final inventory

In [ ]:
result = invoke(LAMBDA_INVENTORY, {'status': 'all'})
summary = result['summary']
print('Final inventory summary:')
for k, v in summary.items(): print(f'  {k}: {v}')
print()
rows = [[c_.get('domain'), c_.get('priority',''), c_.get('renewal_status',''),
         c_.get('days_remaining','')] for c_ in result['certificates']]
print(tabulate(rows, headers=['Domain','Priority','Status','Days Left'], tablefmt='github'))

## Verify keys in Secrets Manager

In [ ]:
for rn in renewal_results:
    path = f'{CERT_SECRETS_PREFIX}/{rn["domain"]}/private-key'
    try:
        s = sm.get_secret_value(SecretId=path)
        d = json.loads(s['SecretString'])
        print(f'  ✅ {rn["domain"]} — key stored at {d.get("generated_at","")[:19]}')
    except sm.exceptions.ResourceNotFoundException:
        print(f'  ❌ {rn["domain"]} — not found')

## Lab 02 Complete

Full lifecycle: scan -> renew (CSR + private key) -> issuance -> inventory.

**Next:** `03_bedrock_agent.ipynb`